# Bacterial growth homeostasis as a Bayesian bandit — the analysis, in code

This notebook reproduces, from the repository's own data, everything shown in the HTML pages:

1. the seven phenotypic variables of one growth cycle and their identities,
2. their coefficients of variation, within-cell and across-generation correlations, and lineage effects, per experiment,
3. sister versus neighbour lineages,
4. two "laws" that fall out of adding the fold ratio $e^{\varphi}$,
5. a hierarchical Thompson-sampling adder model, simulated,
6. the model fitted to one experiment on normalised statistics and validated on held-out ones,
7. the cross-lab transfer test,
8. a summary of what was shown and what was not,
9. the fixes made after an adversarial review, and the full refit.

Every number and figure below is produced by the code in this notebook. The only things imported from the cloned repository are its variable names and dataset lists (`AnalysisCode/global_variables.py`); the processed `csv` files are read directly.

In [ ]:
# --- Setup -------------------------------------------------------------------
import os, sys, json, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import t as student_t, ks_2samp, skew
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

REPO = os.path.abspath('..')          # path of the cloned repository (this notebook lives in BayesianBanditModel/)
sys.path.insert(0, REPO)
# The repository's global_variables.py refuses to run on Linux (it only knows Windows and Mac path separators).
# We load its source with that one check pointed at 'darwin' (which also uses '/'), everything else untouched.
import types
src = open(f'{REPO}/AnalysisCode/global_variables.py').read().replace('from sys import platform', "platform = 'darwin'")
gv = types.ModuleType('global_variables'); exec(src, gv.__dict__)
sm_datasets, tanouchi_datasets, cgsc_6300_wang_exps, lexA3_wang_exps = gv.sm_datasets, gv.tanouchi_datasets, gv.cgsc_6300_wang_exps, gv.lexA3_wang_exps
print('repository symbols for the variables:', {k: gv.symbols['physical_units'][k] for k in ('fold_growth','division_ratio','added_length','generationtime','growth_rate')})

# The seven variables of a cycle, plus the fold ratio e^phi which we add ourselves
V   = ['fold_growth','division_ratio','added_length','generationtime','length_birth','length_final','growth_rate','efold']
SYM = dict(zip(V, ['φ','f','Δ','τ','x0','xτ','α','e^φ']))
LOG = {'added_length','generationtime','length_birth','length_final','growth_rate','efold'}   # log these before z-scoring

# Short labels for the 16 experiments (the empty Br SJ108 dataset is skipped)
EXPS = {'Pooled_SM':'SM', 'Maryam_LongTraces':'MM Maryam', 'MG1655_inLB_LongTraces':'MM MG1655', 'lambda_LB':'MM λ'}
EXPS.update({d: 'Tan ' + d[7:10] for d in tanouchi_datasets})
EXPS.update({d: 'Wang ' + d[4:8] for d in cgsc_6300_wang_exps})
EXPS.update({d: 'lexA3 ' + d[4:8] for d in lexA3_wang_exps})
print(len(EXPS), 'experiments')

## 1. One growth cycle and its variables

Inside one cycle the cell's length grows exponentially, so $\ln(\text{length})$ is a straight line in time. The repository fits that line for every cycle; its slope is the growth rate $\alpha$ and its end points give $x_0$ and $x_\tau$. Everything else is arithmetic:

$$\varphi=\ln\frac{x_\tau}{x_0}=\alpha\tau,\qquad \Delta=x_\tau-x_0,\qquad f=\frac{x_0}{x_\tau^{\text{mother}}},\qquad e^{\varphi}=\frac{x_\tau}{x_0},\qquad r=f\,e^{\varphi}$$

Balanced growth means a daughter is born, on average, the size her mother was: $\langle \ln r\rangle = 0$.

In [ ]:
# --- Load one experiment's processed data and show the first cycles ------------
def load(ds):
    # 'z_score_under_3' = the repository's version with lineage-specific 3-sigma outliers set to NaN
    d = pd.read_csv(f'{REPO}/Datasets/{ds}/ProcessedData/z_score_under_3/physical_units_without_outliers.csv')
    if 'trap_ID' not in d:                      # mother-machine files have one lineage per trap
        d['trap_ID'] = d['lineage_ID']; d['trace'] = 'A'; d['dataset'] = 'MM'
    d['lin']   = d.lineage_ID.astype(str) + '|' + d.trap_ID.astype(str) + '|' + d.trace   # unique lineage key
    d['efold'] = np.exp(d.fold_growth)                                                      # the added variable
    return d

d = load('Maryam_LongTraces')
d[['generation','length_birth','length_final','added_length','generationtime','growth_rate','fold_growth','division_ratio','efold']].head(6).round(3)

In [ ]:
# --- Draw one lineage: raw length versus time, with the per-cycle fits ----------
raw = pd.read_csv(f'{REPO}/Datasets/Maryam_LongTraces/raw_data_all_in_one.csv')
lin = raw[(raw.lineage_ID == 1) & (raw.time < 25)].sort_values('time')
cyc = d[(d.lineage_ID == 1) & (d.end_time < 25)]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(lin.time, lin.length, '.', ms=3, color='0.6', label='measured length')
for _, c in cyc.iterrows():                              # exponential fit of each cycle, from the processed variables
    t = np.linspace(c.start_time, c.end_time, 10)
    ax.plot(t, c.length_birth * np.exp(c.growth_rate * (t - c.start_time)), color='teal', lw=1.5)
ax.set(xlim=(0, 25), xlabel='time (h)', ylabel='length (µm)', title='One lineage: each teal segment is one cycle, x0 → xτ, slope α')
ax.legend(frameon=False); plt.show()

In [ ]:
# --- Check the identities hold in the processed data ---------------------------
print('phi  = alpha*tau       max |error| =', np.nanmax(np.abs(d.fold_growth - d.growth_rate * d.generationtime)).round(12))
print('delta = x_tau - x0     max |error| =', np.nanmax(np.abs(d.added_length - (d.length_final - d.length_birth))).round(12))
print('mean of ln r = ln(f e^phi):', np.nanmean(np.log(d.division_ratio * d.efold)).round(3), ' (balanced growth → 0)')

## 2. The statistics

Five scale-free quantities, computed per experiment. Correlations are computed **after subtracting each lineage's own mean** ("lineage-centred"), so that slow differences between lineages do not masquerade as cell-to-cell structure.

| statistic | formula |
|---|---|
| coefficient of variation | $\mathrm{CV}=\sigma_x/\langle x\rangle$ |
| correlation (same cell) | $r_{xy}=\mathrm{cov}(x,y)/(\sigma_x\sigma_y)$ |
| autocorrelation at lag $k$ | $r_k=\mathrm{corr}(x_n,x_{n+k})$ |
| lineage variance share | $1-\mathrm{Var}(x-\bar x_{\text{lineage}})/\mathrm{Var}(x)$ |
| z-score quantiles | $z=(\ln x-\text{mean})/\text{sd}$, then the 5/25/50/75/95 % points |

In [ ]:
# --- Statistics functions ------------------------------------------------------
def centre(d):
    'subtract each lineage\'s mean from every variable (lineage-centred copy)'
    tc = d.copy(); tc[V] = d.groupby('lin')[V].transform(lambda s: s - s.mean()); return tc

def acf(tc, v, k=1):
    'autocorrelation at lag k along lineages, pooled over all lineages'
    y = tc.groupby('lin')[v].shift(-k); m = tc[v].notna() & y.notna()
    return np.corrcoef(tc[v][m], y[m])[0, 1]

def experiment_stats(d):
    tc = centre(d); out = {}
    for v in V:
        out[('CV', v)]       = d[v].std() / d[v].mean()
        out[('lineage', v)]  = 1 - tc[v].var() / d[v].var()
        for k in (1, 2, 3): out[(f'acf{k}', v)] = acf(tc, v, k)
    out['corr']  = tc[V].corr()                                   # within-cell, lineage-centred
    out['ta_cv'] = {v: d.groupby('lin')[v].mean().std() / d[v].mean() for v in V}
    out['n_cells'], out['n_lin'] = len(d), d.lin.nunique()
    return out

DATA = {ds: load(ds) for ds in EXPS}
STATS = {ds: experiment_stats(DATA[ds]) for ds in EXPS}
print('cycles:', sum(s['n_cells'] for s in STATS.values()), ' lineages:', sum(s['n_lin'] for s in STATS.values()))

In [ ]:
# --- Table: pooled mean, CV, CV of lineage means, lag-1 autocorrelation --------
allD = pd.concat(DATA.values())
rows = []
for v in V[:-1]:
    rows.append([SYM[v], allD[v].mean(), allD[v].std()/allD[v].mean(),
                 np.mean([STATS[ds]['ta_cv'][v] for ds in EXPS]), STATS['Pooled_SM'][('acf1', v)]])
pd.DataFrame(rows, columns=['variable','mean','CV (pooled)','CV of lineage means','lag-1 corr (SM)']).round(3)

In [ ]:
# --- Figure: CV, lag-1 autocorrelation and lineage share per experiment ---------
def strip(ax, key, title, zero=False):
    for i, v in enumerate(V[:-1]):
        vals = [STATS[ds][(key, v)] for ds in EXPS]
        ax.scatter(vals, [i]*len(vals), s=14, color='0.55', zorder=2)
        ax.scatter([STATS['Pooled_SM'][(key, v)]], [i], s=36, color='teal', zorder=3)
        ax.plot([min(vals), max(vals)], [i, i], color='0.8', lw=3, zorder=1)
    ax.set_yticks(range(len(V)-1)); ax.set_yticklabels([SYM[v] for v in V[:-1]]); ax.set_title(title)
    if zero: ax.axvline(0, color='0.3', lw=1)

fig, axs = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
strip(axs[0], 'CV', 'coefficient of variation'); strip(axs[1], 'acf1', 'lag-1 autocorrelation', zero=True); strip(axs[2], 'lineage', 'lineage share of variance')
plt.suptitle('one grey dot per experiment, sister machine in teal', y=1.03); plt.tight_layout(); plt.show()

**Reading the figure.** The division ratio $f$ is the most precisely controlled variable everywhere (CV ≈ 0.08). Sizes have memory (lag‑1 ≈ 0.4–0.7: a pure adder predicts 0.5). $\varphi$, $\tau$ and $e^{\varphi}$ are *negatively* autocorrelated in every experiment — the signature of a correction: a cell that grew too much hands its daughter a short cycle. Lineage effects are small in the long mother-machine traces and large in the 25-generation sister machine, where lineages have not had time to forget their channel.

In [ ]:
# --- Within-cell and mother→daughter correlation matrices ---------------------
allTC = pd.concat([centre(DATA[ds]) for ds in EXPS])
intra = allTC[V[:-1]].corr()
cross = pd.DataFrame(index=V[:-1], columns=V[:-1], dtype=float)      # row = mother's variable, column = daughter's
for a in V[:-1]:
    for b in V[:-1]:
        y = allTC.groupby('lin')[b].shift(-1); m = allTC[a].notna() & y.notna()
        cross.loc[a, b] = np.corrcoef(allTC[a][m], y[m])[0, 1]

fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, M, title in ((axs[0], intra, 'same cell (lineage-centred)'), (axs[1], cross, 'mother (row) → daughter (column)')):
    im = ax.imshow(M.values.astype(float), cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(7)); ax.set_xticklabels([SYM[v] for v in V[:-1]]); ax.set_yticks(range(7)); ax.set_yticklabels([SYM[v] for v in V[:-1]])
    for i in range(7):
        for j in range(7): ax.text(j, i, f'{M.values[i, j]:.2f}', ha='center', va='center', fontsize=7.5, color='white' if abs(M.values[i, j]) > .55 else 'black')
    ax.set_title(title)
plt.tight_layout(); plt.show()

Three structural facts in the left panel: $\varphi$–$\tau$ (+0.85) and $\varphi$–$\Delta$ (+0.73) because fold growth is set by how long the cell waits; $x_0$–$\varphi$ (−0.39) is the adder, cells born large grow fewer folds; and $\tau$–$\alpha$ (−0.28) shows that $\tau$ absorbs growth-rate fluctuations so that $\varphi$ does not ($\alpha$–$\varphi$ ≈ 0). In the right panel the two strong entries are bookkeeping ($x_0' = f x_\tau$); the negative entries into $\varphi'$ and $\tau'$ are the correction.

## 3. Sisters versus neighbours

The sister machine has two kinds of pair in one channel: **sister lineages** (SL, same mother) and **neighbour lineages** (NL, no shared mother). Comparing them separates inheritance from shared environment.

In [ ]:
# --- A-B correlation by generation, SL vs NL, and over time-averages ------------
sm = DATA['Pooled_SM']
def pair_corr(dd, v, gen=None):
    'correlation between lineage A and B of the same trap; gen=None uses lineage time-averages'
    if gen is None: g = dd.groupby(['trap_ID', 'trace'])[v].mean().unstack()
    else:           g = dd[dd.generation == gen].pivot_table(index='trap_ID', columns='trace', values=v)
    return g['A'].corr(g['B'])

fig, ax = plt.subplots(figsize=(7, 3.3))
for v, c in (('generationtime', 'teal'), ('growth_rate', 'darkorange'), ('length_birth', '0.3')):
    for ds, ls in (('SL', '-'), ('NL', '--')):
        ax.plot(range(12), [pair_corr(sm[sm.dataset == ds], v, g) for g in range(12)], ls, color=c, label=f'{SYM[v]} {ds}')
ax.axhline(0, color='0.5', lw=.8); ax.set(xlabel='generations since the pair was born', ylabel='corr(A, B)'); ax.legend(ncol=3, frameon=False); plt.show()

print('time-average correlation A vs B:')
print(pd.DataFrame({ds: {SYM[v]: round(pair_corr(sm[sm.dataset == ds], v), 2) for v in V[:-1]} for ds in ('SL', 'NL')}).T)

Sisters start strongly synchronised and decay to the neighbour floor within two to three generations; the floor (α ≈ 0.3–0.4) is the channel. Over 25-generation time averages sisters still exceed neighbours (α 0.81 vs 0.65, τ 0.86 vs 0.59): a second, much slower inherited component. Two inherited timescales.

## 4. Adding $e^{\varphi}$: two laws with no free parameters

From $x_0' = f\,(x_0 + \Delta)$ and a policy $\Delta = \bar\Delta + k\,(x_0-\bar x_0)$ one gets, in one line each,
$$\langle e^{\varphi}\rangle = 1/\langle f\rangle \qquad\text{and}\qquad r_1(x_0) = \langle f\rangle\,(1+k).$$
$k$ is the slope of $\Delta$ on $x_0$ within lineages: $k=0$ is a pure adder, $k<0$ leans to a size target, $k>0$ to a time target.

In [ ]:
# --- Measure <f>, <e^phi>, k and the lag-1 memory of x0 in every experiment ------
rows = []
for ds in EXPS:
    d = DATA[ds]; tc = centre(d); m = tc.added_length.notna() & tc.length_birth.notna()
    k = np.polyfit(tc.length_birth[m], tc.added_length[m], 1)[0]
    rows.append([EXPS[ds], d.division_ratio.mean(), 1/d.division_ratio.mean(), d.efold.mean(), k, d.division_ratio.mean()*(1+k), STATS[ds][('acf1', 'length_birth')]])
laws = pd.DataFrame(rows, columns=['exp', '<f>', '1/<f>', '<e^φ>', 'k', '<f>(1+k)', 'acf1 x0']).set_index('exp')

fig, axs = plt.subplots(1, 2, figsize=(8, 3.6))
axs[0].scatter(laws['1/<f>'], laws['<e^φ>'], color='teal'); axs[0].plot([1.8, 2.3], [1.8, 2.3], 'k--', lw=.8); axs[0].set(xlabel='1 / <f>', ylabel='<e^φ>', title='fold target = inverse asymmetry')
axs[1].scatter(laws['<f>(1+k)'], laws['acf1 x0'], color='darkorange'); axs[1].plot([.3, .75], [.3, .75], 'k--', lw=.8); axs[1].set(xlabel='<f>(1+k)', ylabel='lag-1 autocorrelation of x0', title='size memory = f(1+k)')
plt.tight_layout(); plt.show()
laws.round(3)

Both laws hold within 0.03 in 15 of 16 experiments (Wang 0512, the noisiest, is the exception). Wang's long size memory therefore comes from $k\approx+0.2$, not from a hidden inherited size state. Distribution shapes add one more fact: $\varphi$ is nearly symmetric while $\Delta$ is heavy-tailed, so the model below uses a Student-t, not a Gaussian, for $\Delta$.

In [ ]:
# --- Skewness of phi, e^phi, delta and ln(delta): phi is the Gaussian variable ----
pd.DataFrame({EXPS[ds]: {'φ': skew(DATA[ds].fold_growth.dropna()), 'e^φ': skew(DATA[ds].efold.dropna()),
                          'Δ': skew(DATA[ds].added_length.dropna()), 'ln Δ': skew(np.log(DATA[ds].added_length.dropna()))} for ds in EXPS}).T.round(2)

## 5. The model

Each generation the cell "pulls three arms" — a growth rate, an amount to add and a division ratio — by drawing them from its current belief about the environment (Thompson sampling). The belief is inherited from the mother and drifts.

**Layer 1, the belief and how it is inherited** (AR(1) processes; $\rho$ is the fraction kept per generation)
$$s_n=\rho_{\text{slow}}s_{n-1}+\varepsilon_s,\qquad a_n=\rho_{\text{fast}}a_{n-1}+\varepsilon_a,\qquad e=\text{constant per channel}$$

**Layer 2, pulling the arms**
$$\ln\alpha_n=\ln\mu_\alpha+e_\alpha+s_{\alpha,n}+a_n,\qquad
\Delta_n=\big[\bar\Delta\,e^{e_\Delta+s_{\Delta,n}}+k\,(x_0-\bar x_0)\big]\,e^{\sigma_\Delta t_\nu},\qquad
f_n=\langle f\rangle+e_f+\text{small AR(1)}$$

**Layer 3, the arithmetic** (no free parameters)
$$x_\tau=x_0+\Delta_n,\quad \varphi_n=\ln(x_\tau/x_0),\quad \tau_n=\varphi_n/\alpha_n,\quad x_0^{\text{daughter}}=f_n\,x_\tau$$

| symbol | meaning |
|---|---|
| $s_n$ | **s**low state: long-term estimate of growth rate / added length, copied to both daughters |
| $a_n$ | f**a**st state on growth rate: what a daughter still carries of her mother's deviation |
| $e$ | **e**nvironment offset of the channel, shared by both lineages in it |
| $\varepsilon, t_\nu$ | the dice: Gaussian kicks, and a Student-t draw with $\nu$ degrees of freedom (heavy tails) |
| $\mu_\alpha,\bar\Delta,\langle f\rangle,\bar x_0,k$ | measured from the data, never fitted |
| $\rho,\sigma,\nu$ | fitted constants |

In [ ]:
# --- The simulator: every lineage of an experiment, one generation per loop step -------
PNAMES = ['s_env_a','s_inh_a','rho_slow','rho_fast','s_fast_a','s_d','s_lin_d','s_f','s_lin_f','s_meas','log_nu_d','s_fast_d','rho_fast_d','log_nu_a','rho_f']
P0 = np.array([0.08,0.06,0.97,0.4,0.11,0.21,0.08,0.045,0.02,0.03,np.log(4),0.0,0.3,np.log(5),0.1])          # starting point
LO = np.array([0,0,0.5,-0.3,0.01,0.03,0,0.01,0,0.005,np.log(2.2),0,0,np.log(2.2),-0.3])                      # lower bounds
HI = np.array([0.3,0.3,0.999,0.9,0.4,0.6,0.3,0.12,0.08,0.12,np.log(60),0.3,0.9,np.log(60),0.7])              # upper bounds

def simulate(theta, fixed, n_lin, gens, trap, trace, seed=0):
    p = dict(zip(PNAMES, theta)); rng = np.random.default_rng(seed)
    mu_a, mu_d, f0, k = fixed['mu_a'], fixed['mu_d'], fixed['f0'], fixed['k']
    nu_d, nu_a = np.exp(p['log_nu_d']), np.exp(p['log_nu_a']); sis = fixed['sisters']
    traps, tinv = np.unique(trap, return_inverse=True); nT = len(traps)
    # Layer 1a: channel offsets e (one draw per trap, shared by its two lineages)
    env_a = rng.normal(0, p['s_env_a'], nT)[tinv]; env_d = rng.normal(0, p['s_env_a']*1.2, nT)[tinv]; env_f = rng.normal(0, p['s_f']*0.3, nT)[tinv]
    lin_f = rng.normal(0, p['s_lin_f'], n_lin); lin_d = rng.normal(0, p['s_lin_d'], n_lin)
    # Layer 1b: inherited states at birth; sisters share the mother's draw, neighbours do not
    sl_a = rng.normal(0, p['s_inh_a'], nT)[tinv] if sis else rng.normal(0, p['s_inh_a'], n_lin)
    fa   = rng.normal(0, p['s_fast_a'], nT)[tinv] if sis else rng.normal(0, p['s_fast_a'], n_lin)
    fd = np.zeros(n_lin); fstate = np.zeros(n_lin)
    x0bar = mu_d*f0/(1-f0); mother = x0bar/f0*np.exp(rng.normal(0, 0.12, nT))[tinv]
    fprev = np.clip(rng.normal(f0+env_f, p['s_f'], n_lin), 0.3, 0.7)
    if sis: fA = np.clip(rng.normal(f0, p['s_f'], nT), 0.3, 0.7)[tinv]; fprev = np.where(trace == 'A', fA, 1-fA)
    x0 = fprev*mother
    out = {v: np.full((n_lin, gens), np.nan) for v in V}
    q_s, q_f, q_fd = np.sqrt(1-p['rho_slow']**2), np.sqrt(1-p['rho_fast']**2), np.sqrt(1-p['rho_fast_d']**2)
    for g in range(gens):
        # Layer 1: AR(1) updates of the belief states
        sl_a = p['rho_slow']*sl_a + rng.normal(0, p['s_inh_a']*q_s, n_lin)
        fa   = p['rho_fast']*fa   + student_t.rvs(nu_a, size=n_lin, random_state=rng)*np.sqrt((nu_a-2)/nu_a)*p['s_fast_a']*q_f
        fd   = p['rho_fast_d']*fd + rng.normal(0, p['s_fast_d']*q_fd, n_lin)
        # Layer 2: pull the arms
        alpha = mu_a*np.exp(env_a + sl_a + fa)
        noise = student_t.rvs(nu_d, size=n_lin, random_state=rng)*p['s_d']*np.sqrt((nu_d-2)/nu_d)
        delta = np.maximum((mu_d*np.exp(env_d + lin_d + fd) + k*(x0 - x0bar))*np.exp(noise), 0.15*mu_d)
        # Layer 3: arithmetic, then measurement noise on the two lengths
        xt = x0 + delta; phi = np.log(xt/x0); tau = phi/alpha
        x0m = x0*np.exp(rng.normal(0, p['s_meas'], n_lin)); xtm = xt*np.exp(rng.normal(0, p['s_meas'], n_lin))
        out['fold_growth'][:, g] = np.log(xtm/x0m); out['efold'][:, g] = xtm/x0m; out['added_length'][:, g] = xtm - x0m
        out['generationtime'][:, g] = tau; out['length_birth'][:, g] = x0m; out['length_final'][:, g] = xtm
        out['growth_rate'][:, g] = np.log(xtm/x0m)/tau; out['division_ratio'][:, g] = fprev if g > 0 else np.nan
        fstate = p['rho_f']*fstate + rng.normal(0, p['s_f']*np.sqrt(1-p['rho_f']**2), n_lin)
        f = np.clip(f0 + env_f + lin_f + fstate, 0.25, 0.75); x0 = f*xt; fprev = f
    return out

## 6. Fitting on normalised statistics, validating on held-out ones

The model's likelihood is intractable (layer 3 is a deterministic map), so the fit is simulation-based:
$$L(\theta)=\sum_i w_i\,\big(\hat s_i - s_i(\theta)\big)^2$$
where $\hat s_i$ are the data's scale-free statistics (CVs, lineage-centred correlations, lag‑1/2 autocorrelations, lineage shares, z-score quantiles, sister time-average correlations) and $s_i(\theta)$ the same statistics from a simulation with a **fixed seed**. $L$ is minimised by Nelder–Mead. Held out and never seen by the optimiser: lag‑3 autocorrelations, mother→daughter correlations, the adder slope, the CV of lineage-mean $r$, sister correlations at birth, and the physical-unit means.

Below, the fit is run on the sister-machine experiment with a short optimisation so the notebook runs in about a minute; set `MAXFEV` higher (the HTML used 1,800) for the full result.

In [ ]:
# --- Statistics computed on a (n_lineages × generations) matrix, for data and model alike ----
QS = [0.05, 0.25, 0.5, 0.75, 0.95]

def to_matrix(d):
    lins = d.lin.unique(); idx = {l: i for i, l in enumerate(lins)}
    G = int(d.groupby('lin').size().max()); M = {v: np.full((len(lins), G), np.nan) for v in V}
    li = d.lin.map(idx).values; gi = d.generation.values.astype(int)
    for v in V: M[v][li, gi] = d[v].values
    meta = d.groupby('lin').first()[['trap_ID', 'trace']].loc[lins]
    return M, meta.trap_ID.values, meta.trace.values

def zref(M):
    'the data\'s mean and sd of each (logged) variable, used to z-score both data and model'
    ref = {}
    for v in V:
        x = M[v][~np.isnan(M[v])]; x = np.log(x[x > 0]) if v in LOG else x
        ref[v] = (x.mean(), x.std())
    return ref

def matrix_stats(M, trap, trace, ref):
    out = {}; tc = {}
    for v in V:
        X = M[v]; tc[v] = X - np.nanmean(X, axis=1, keepdims=True)
        out[f'cv_{v}'] = np.nanstd(X)/np.nanmean(X); out[f'fracL_{v}'] = 1 - np.nanvar(tc[v])/np.nanvar(X)
        for k in (1, 2, 3):
            a, b = tc[v][:, :-k], tc[v][:, k:]; m = ~np.isnan(a) & ~np.isnan(b)
            out[f'acf{k}_{v}'] = np.corrcoef(a[m], b[m])[0, 1]
        x = X[~np.isnan(X)]; x = np.log(x[x > 0]) if v in LOG else x
        for q, val in zip(QS, np.quantile((x - ref[v][0])/ref[v][1], QS)): out[f'zq_{v}_{q}'] = val
    for i, a in enumerate(V):
        for b in V[i+1:]:
            m = ~np.isnan(tc[a]) & ~np.isnan(tc[b]); out[f'corr_{a}_{b}'] = np.corrcoef(tc[a][m], tc[b][m])[0, 1]
    for a in ('length_final', 'added_length', 'length_birth'):                         # held out: mother → daughter
        for b in ('fold_growth', 'length_birth', 'generationtime'):
            x, y = tc[a][:, :-1], tc[b][:, 1:]; m = ~np.isnan(x) & ~np.isnan(y); out[f'cross_{a}_{b}'] = np.corrcoef(x[m], y[m])[0, 1]
    x, y = M['length_birth'], M['length_final']; m = ~np.isnan(x) & ~np.isnan(y); out['adder_slope'] = np.polyfit(x[m], y[m], 1)[0]
    rm = np.nanmean(M['division_ratio']*M['efold'], axis=1); out['cv_lineage_r'] = np.nanstd(rm)/np.nanmean(rm)
    A = np.where(trace == 'A')[0]; B = np.where(trace == 'B')[0]; ta = {t: i for t, i in zip(trap[A], A)}
    pairs = [(ta[t], j) for t, j in zip(trap[B], B) if t in ta]
    if len(pairs) > 10:
        ia, ib = map(list, zip(*pairs))
        for v in ('growth_rate', 'generationtime', 'length_birth', 'added_length', 'fold_growth'):
            ma, mb = np.nanmean(M[v][ia], axis=1), np.nanmean(M[v][ib], axis=1); out[f'pairTA_{v}'] = np.corrcoef(ma, mb)[0, 1]
            g0a, g0b = M[v][ia, 0], M[v][ib, 0]; m = ~np.isnan(g0a) & ~np.isnan(g0b); out[f'pairG0_{v}'] = np.corrcoef(g0a[m], g0b[m])[0, 1]
    return out

is_fit  = lambda k: k.split('_')[0] in ('cv', 'fracL', 'acf1', 'acf2', 'corr', 'zq') or k.startswith('pairTA')
is_hold = lambda k: k.startswith('acf3') or k.startswith('cross') or k.startswith('pairG0') or k in ('adder_slope', 'cv_lineage_r')

In [ ]:
# --- Fit the sister-machine experiment ----------------------------------------
MAXFEV = 300                                   # 1800 in the HTML; raise for a tighter fit
ds = 'Pooled_SM'; d = DATA[ds]; M, trap, trace = to_matrix(d); n_lin = M['fold_growth'].shape[0]
gens = int(np.nanmedian(np.sum(~np.isnan(M['fold_growth']), axis=1)))

# measured, not fitted
tc = centre(d); m = tc.added_length.notna() & tc.length_birth.notna()
fixed = dict(mu_a=float(np.exp(np.nanmean(np.log(M['growth_rate'])))), mu_d=float(np.nanmean(M['added_length'])),
             f0=float(np.nanmean(M['division_ratio'])), k=float(np.polyfit(tc.length_birth[m], tc.added_length[m], 1)[0]), sisters=True)
print('measured inputs:', {k: round(v, 3) for k, v in fixed.items() if k != 'sisters'})

ref = zref(M); S = matrix_stats(M, trap, trace, ref)
keys = [k for k in S if is_fit(k)]; hold = [k for k in S if is_hold(k)]
tgt = np.array([S[k] for k in keys]); w = np.array([3. if k.startswith('cv') else (.5 if k.startswith('zq') else 1.) for k in keys])

def loss(th):
    th = np.clip(th, LO, HI); s = matrix_stats(simulate(th, fixed, n_lin, gens, trap, trace, seed=0), trap, trace, ref)
    r = np.array([s.get(k, np.nan) for k in keys]) - tgt; r[np.isnan(r)] = 0
    return float(np.sum(w*r**2))

print('loss at starting point:', round(loss(P0), 3))
best = minimize(loss, P0, method='Nelder-Mead', options={'maxfev': MAXFEV, 'xatol': 1e-3, 'fatol': 1e-4})
theta = np.clip(best.x, LO, HI)
print('loss after fit:', round(best.fun, 3), ' evaluations:', best.nfev)
pd.Series(dict(zip(PNAMES, theta))).round(3)

In [ ]:
# --- Validation 1: held-out statistics with a NEW seed -------------------------
sim = simulate(theta, fixed, n_lin, gens, trap, trace, seed=7); s = matrix_stats(sim, trap, trace, ref)
fit_pairs  = np.array([(S[k], s[k]) for k in keys if not np.isnan(s.get(k, np.nan))])
hold_pairs = np.array([(S[k], s[k]) for k in hold if not np.isnan(s.get(k, np.nan))])
rmse = lambda P: np.sqrt(np.mean((P[:, 0] - P[:, 1])**2))
print(f'fitted statistics:  n={len(fit_pairs)}  rmse={rmse(fit_pairs):.3f}')
print(f'held-out statistics: n={len(hold_pairs)}  rmse={rmse(hold_pairs):.3f}  corr={np.corrcoef(hold_pairs[:,0], hold_pairs[:,1])[0,1]:.3f}')

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.scatter(fit_pairs[:, 0], fit_pairs[:, 1], s=10, color='0.6', label='fitted')
ax.scatter(hold_pairs[:, 0], hold_pairs[:, 1], s=22, color='darkorange', label='held out')
ax.plot([-1, 1.2], [-1, 1.2], 'k--', lw=.8); ax.set(xlabel='data', ylabel='model', title='sister machine'); ax.legend(frameon=False); plt.show()

print('worst held-out:'); print(pd.DataFrame([(k, S[k], s[k]) for k in hold], columns=['statistic', 'data', 'model']).assign(gap=lambda x: (x.data - x.model).abs()).sort_values('gap', ascending=False).head(5).round(2).to_string(index=False))

In [ ]:
# --- Validation 2: distribution shape in z-units (normalised), and physical means --------------
fig, ax = plt.subplots(figsize=(7, 3.2))
for i, v in enumerate(V[:-1]):
    x = M[v][~np.isnan(M[v])]; y = sim[v][~np.isnan(sim[v])]
    if v in LOG: x, y = np.log(x[x > 0]), np.log(y[y > 0])
    zx, zy = (x - ref[v][0])/ref[v][1], (y - ref[v][0])/ref[v][1]
    ax.plot(np.quantile(zx, QS), [i]*5, '|', ms=14, mew=2, color='teal'); ax.plot(np.quantile(zy, QS), [i]*5, 'o', ms=5, color='darkorange')
    ax.text(2.3, i, f'KS {ks_2samp(zx, zy).statistic:.2f}', va='center', fontsize=8)
ax.set_yticks(range(7)); ax.set_yticklabels([SYM[v] for v in V[:-1]]); ax.set(xlabel='z-score (data mean and sd)', title='quantiles 5/25/50/75/95 %: data (teal ticks) vs model (orange)'); plt.show()

print('physical units, data vs model:')
print(pd.DataFrame({SYM[v]: [np.nanmean(M[v]), np.nanmean(sim[v])] for v in ('length_birth', 'generationtime', 'added_length', 'growth_rate')}, index=['data', 'model']).round(3))

## 7. The cross-lab transfer test

The strongest check: freeze the per-generation **rule** (noise widths $\sigma_{\text{fast},\alpha},\sigma_\Delta,\sigma_f$, tail exponents $\nu_\Delta,\nu_\alpha$, short memories $\rho_{\text{fast}},\rho_f$) at the median of the fits from the *other* labs (Wang 2010 and Tanouchi 2015, twelve experiments), and refit only the six **environment-structure** parameters (trap and lineage variances, $\rho_{\text{slow}}$, measurement noise) on this lab's sister-machine data. If the held-out statistics are still predicted, the rule is not a property of one lab.

The per-experiment fits come from `fit_results.json`, produced by `fit_bandit_model.py` in this folder (1,800 evaluations per experiment).

In [ ]:
# --- Freeze the rule from the other labs, refit only the environment structure ---------------
FULL = json.load(open('fit_results.json'))                      # per-experiment fits from fit_bandit_model.py
RULE = ['rho_fast','s_fast_a','s_d','log_nu_d','log_nu_a','s_f','rho_f','s_fast_d','rho_fast_d']
FREE = [p for p in PNAMES if p not in RULE]
this_lab = ['Pooled_SM','Maryam_LongTraces','MG1655_inLB_LongTraces','lambda_LB']
others   = [e for e in EXPS if e not in this_lab]

rule_others  = {p: np.median([FULL[e]['params'][p] for e in others])   for p in RULE}
rule_thislab = {p: np.median([FULL[e]['params'][p] for e in this_lab]) for p in RULE}
show = lambda r: {p: round(float(np.exp(r[p]) if p.startswith('log') else r[p]), 2) for p in RULE}
print('rule learned from Wang + Tanouchi:', show(rule_others)); print('rule learned from this lab:        ', show(rule_thislab))

base = np.array([rule_others[p] if p in rule_others else FULL[ds]['params'][p] for p in PNAMES])
fi = [PNAMES.index(p) for p in FREE]
def unpack(x): th = base.copy(); th[fi] = x; return np.clip(th, LO, HI)
def loss_free(x):
    s = matrix_stats(simulate(unpack(x), fixed, n_lin, gens, trap, trace, seed=0), trap, trace, ref)
    r = np.array([s.get(k, np.nan) for k in keys]) - tgt; r[np.isnan(r)] = 0; return float(np.sum(w*r**2))
bestT = minimize(loss_free, base[fi], method='Nelder-Mead', options={'maxfev': 200, 'xatol': 1e-3, 'fatol': 1e-4})
thetaT = unpack(bestT.x)

simT = simulate(thetaT, fixed, n_lin, gens, trap, trace, seed=7); sT = matrix_stats(simT, trap, trace, ref)
holdT = np.array([(S[k], sT[k]) for k in hold if not np.isnan(sT.get(k, np.nan))])
print(f'\nheld-out rmse on the sister machine: transferred rule {rmse(holdT):.3f}   own fit (this notebook) {rmse(hold_pairs):.3f}   own full fit (fit_results.json) {np.sqrt(np.mean([(a-b)**2 for a,b in FULL[ds]["holdout"].values() if a==a and b==b])):.3f}')

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.scatter(hold_pairs[:, 0], hold_pairs[:, 1], s=26, color='darkorange', label='own fit')
ax.scatter(holdT[:, 0], holdT[:, 1], s=26, marker='s', facecolor='none', edgecolor='teal', label='rule frozen from other labs')
ax.plot([-.6, 1], [-.6, 1], 'k--', lw=.8); ax.set(xlabel='data', ylabel='model', title='held-out statistics, sister machine'); ax.legend(frameon=False); plt.show()

The fit reproduces the CVs, within-cell correlations and distribution shapes (KS ≲ 0.05, except $\tau$, which is frame-quantised in the data) and predicts the held-out statistics to within about ±0.1. The largest held-out miss is the same one as in the HTML: sisters' first-generation timing is more synchronised than shared size and growth rate can explain.

## 8. What this did and did not show

**Did:** per experiment, per lineage and pooled, the model reproduces the coefficients of variation, the within-cell and across-generation correlations, the lineage shares and the distribution shapes, in normalised units and back in physical units, with one measured pair $(\langle f\rangle, k)$ per environment and noise widths that barely move between labs. The two laws of section 4 hold without any fitting.

**Did, after the transfer test:** the per-generation rule learned from the other labs predicts this lab's held-out statistics nearly as well as a direct fit, and the rules learned from either side agree to two decimals; only the growth-rate noise width is lab-specific.

**Did not:** sister synchrony at birth is under-predicted; one experiment with filamentous cells is not reproduced; the mother/daughter length mismatch in the older datasets is attributed to measurement noise but not modelled; $\rho_{\text{slow}}$ is poorly identified from single lineages; the fit is moment-matching, not a likelihood, and the "bandit" reading is an interpretation the data do not force.

To fit all 16 experiments, loop the cells of section 6 over `EXPS` (the attached `fit_bandit_model.py` does exactly that).

## 9. After the adversarial review

An adversarial review found defects in the analysis above (listed in section 13 of the revision-4 HTML). This section shows the three fixes used in the final refit, with code:

1. **measurement noise is measured from the raw images**, not fitted;
2. **CMA-ES on a seed-averaged loss** replaces single-seed Nelder–Mead;
3. **one parameter set per lab**: experiments from the same lab and strain share their 15 parameters, while each keeps its own measured means, gain $k$ and measurement noise.

It also corrects two claims made in section 4. $\langle e^{\varphi}\rangle = 1/\langle f\rangle$ follows from the lineages being stationary, so it is not a finding. $r_1(x_0)=\langle f\rangle(1+k)$ is close to an identity once $k$ is defined as a regression slope.

In [ ]:
# --- Fix 1: measure length noise from raw images ----------------------------------
# Within a cycle ln(length) is a straight line in time. The raw points scatter around it with sd sigma_pt;
# a fitted end length then has error sigma_pt * sqrt(1/n + (t_end - t_mean)^2 / Sxx). That is the model's s_meas.
def endpoint_noise(ds, n_cycles=800):
    raw = pd.read_csv(f'{REPO}/Datasets/{ds}/raw_data_all_in_one.csv'); pu = pd.read_csv(f'{REPO}/Datasets/{ds}/ProcessedData/physical_units.csv')
    key = [c for c in ('dataset','trap_ID','trace','lineage_ID') if c in raw.columns and c in pu.columns]
    for c in key:
        if c in ('trap_ID','lineage_ID'): raw[c] = raw[c].astype(int); pu = pu[pu[c].notna()]; pu[c] = pu[c].astype(int)
    g = raw[raw.length > 0].groupby(key); pts, ends = [], []
    for _, c in pu.dropna(subset=['start_time','end_time']).sample(min(n_cycles, len(pu)), random_state=0).iterrows():
        try: r = g.get_group(tuple(c[x] for x in key))
        except KeyError: continue
        seg = r[(r.time >= c.start_time - 1e-6) & (r.time <= c.end_time + 1e-6)]
        if len(seg) < 5: continue
        t = seg.time.values; y = np.log(seg.length.values); res = y - np.polyval(np.polyfit(t, y, 1), t)
        s = np.sqrt(np.sum(res**2)/(len(t)-2)); pts.append(s); ends.append(s*np.sqrt(1/len(t) + (t[-1]-t.mean())**2/np.sum((t-t.mean())**2)))
    return np.median(pts), np.median(ends)

for ds in ('Pooled_SM', 'MC4100_27C (Tanouchi 2015)', '20090525_E_coli_MG1655_(CGSC_6300)_Wang2010'):
    sp, se = endpoint_noise(ds); print(f'{EXPS[ds]:10s} per-image scatter {100*sp:.1f}%   error of a fitted end length {100*se:.1f}%')
print('\nNoise this small cannot bias the regression slope k noticeably, so k stays as measured.')

The full refit, all 16 experiments in four lab groups, takes about 45 minutes, so it is run by `review_fixes/fit5.py` and its results are loaded here.

In [ ]:
# --- Load the full refit (fixes 1-3) and tabulate it ------------------------------
RF = 'review_fixes/fit5_results.json' if os.path.exists('review_fixes/fit5_results.json') else 'fit5_results.json'
F5 = json.load(open(RF))
rows = []
for g, r in F5.items():
    for d, e in r['per_experiment'].items():
        rows.append([g, e['label'], e['s_meas'], e['fixed']['k_ols'], e['fit_rmse'], e['hold_rmse'], np.median(list(e['ks'].values()))])
refit = pd.DataFrame(rows, columns=['lab group', 'experiment', 'length error', 'k', 'fit rmse', 'held-out rmse', 'KS median']).round(3)
print(f"parameters: {15*len(F5)} shared (one set per lab) instead of {15*len(EXPS)} per experiment")
print(f"mean held-out rmse: {refit['held-out rmse'].mean():.3f}")
refit

In [ ]:
# --- Fix 2: did the two CMA-ES restarts per lab agree? ----------------------------
show = ['rho_fast', 's_d', 'log_nu_d', 's_inh_a', 'rho_slow', 's_env_a']
conv = {}
for g, r in F5.items():
    a, b = r['run_params']
    conv[g] = {'loss': f"{r['run_losses'][0]:.2f} / {r['run_losses'][1]:.2f}"}
    conv[g].update({p: f"{(np.exp(a[p]) if p.startswith('log') else a[p]):.2f} / {(np.exp(b[p]) if p.startswith('log') else b[p]):.2f}" for p in show})
print('run 1 / run 2  (a parameter is identified only where the two agree)')
pd.DataFrame(conv).T

In [ ]:
# --- Held-out statistics of the refit, pooled over all 16 experiments ---------------
pairs = np.array([p for r in F5.values() for e in r['per_experiment'].values()
                  for k, p in e['pairs'].items() if None not in p and p[0] == p[0] and p[1] == p[1]])
fig, axs = plt.subplots(1, 2, figsize=(9, 3.8))
axs[0].bar(range(len(refit)), refit['held-out rmse'], color='teal'); axs[0].set_xticks(range(len(refit)))
axs[0].set_xticklabels(refit['experiment'], rotation=70, fontsize=7); axs[0].set(ylabel='held-out rmse', title='refit, per experiment')
axs[1].scatter(pairs[:, 0], pairs[:, 1], s=14, color='darkorange'); axs[1].plot([-.2, 1], [-.2, 1], 'k--', lw=.8)
axs[1].set(xlabel='data', ylabel='model', title='sister / neighbour pair correlations'); plt.tight_layout(); plt.show()

**Reading the refit.** Held-out accuracy is somewhat worse than the per-experiment fits of section 6 (mean 0.096 against 0.080), with a quarter of the parameters. The two restarts end at similar losses, but only one parameter agrees everywhere: $\sigma_\Delta\approx0.25$, the added-length noise. The slow memory sits on its lower bound in most groups, and the tail exponent differs between labs (heavy-tailed here, Gaussian in Wang). The "common rule across labs" therefore shrinks to that single number.

**What still is not fixed.** Sister synchrony at birth is real: it survives dropping every cycle that was already under way when filming started (sisters 0.70, neighbours 0.15). An inherited timing state was tried and only partly explains it, so it is left out of this refit. Neighbours share growth rate but barely share timing, which the model cannot reproduce. The heavy-tail exponent is not identified in the sister machine. Error bars and the transfer test have not yet been re-run on the fixed pipeline.